In [1]:
import openpyxl
from pathlib import Path
from collections import defaultdict

In [2]:
def esc(s):
    return s.strip().upper().replace("'", "''") if s else ""

CHUNK = 35
KODE_KAB = "1671"
OUT_COLS = ", ".join(
    f"b.{c}"
    for c in [
        "assignment_id",
        "level_3_full_code",
        "nama_usaha_dicari",
        "nama_usaha",
        "nama_usaha_edit",
        "assignment_status_alias",
        "keberadaan_usaha_label",
        "nama_usaha_sim",
        "nama_usaha_edit_sim",
    ]
)

def build_sql(names, kode_kec):
    blocks = []
    for n in names:
        lit = f"'{esc(n)}'"
        blocks.append(
            f"(SELECT assignment_id, level_3_full_code, {lit} AS nama_usaha_dicari, "
            "nama_usaha, nama_usaha_edit, assignment_status_alias, keberadaan_usaha_label, "
            "nama_usaha_sim, nama_usaha_edit_sim "
            "FROM (SELECT assignment_id, level_3_full_code, nama_usaha, nama_usaha_edit, "
            "assignment_status_alias, keberadaan_usaha_label, "
            f"ngram_search(nama_usaha, {lit}, 2) AS nama_usaha_sim, "
            f"ngram_search(nama_usaha_edit, {lit}, 2) AS nama_usaha_edit_sim "
            "FROM base_data) x "
            "WHERE nama_usaha_sim >= 0.8 OR nama_usaha_edit_sim >= 0.8)"
        )
    union_block = "\n    UNION ALL\n    ".join(blocks)

    # filter kecamatan hanya ditambahkan kalau kode_kec ada isinya
    kec_filter = f"\n    AND a.level_3_full_code = '{kode_kec}'" if kode_kec else ""

    return f"""WITH base_data AS (
  SELECT DISTINCT
    a.assignment_id, a.level_3_full_code,
    nama_usaha, nama_usaha_edit,
    a.assignment_status_alias, keberadaan_usaha_label
  FROM tgr_fd68e454.base_table_assignment a
  LEFT JOIN tgr_fd68e454.root_table b
    ON (a.assignment_id = b.assignment_id)
  LEFT JOIN tgr_fd68e454.se2026_nested c
    ON (a.assignment_id = c.assignment_id)
  WHERE a.is_active = 1
    AND a.level_2_full_code = '{KODE_KAB}'{kec_filter}
)
SELECT {OUT_COLS} FROM (
  SELECT t.assignment_id, t.level_3_full_code, t.nama_usaha_dicari, t.nama_usaha, t.nama_usaha_edit, t.assignment_status_alias, t.keberadaan_usaha_label, t.nama_usaha_sim, t.nama_usaha_edit_sim, ROW_NUMBER() OVER (
    PARTITION BY t.nama_usaha_dicari
    ORDER BY GREATEST(
      t.nama_usaha_sim,
      CASE WHEN t.nama_usaha_edit_sim = '-' THEN t.nama_usaha_sim ELSE t.nama_usaha_edit_sim END
    ) DESC
  ) AS rn
  FROM (
    {union_block}
  ) t
) b
WHERE b.rn = 1;"""

wb = openpyxl.load_workbook(
    "../usaha_sulit_palembang.xlsx", read_only=True, data_only=True
)
ws = wb['Sheet1']
rows = ws.iter_rows(values_only=True)
header = next(rows)
idx_nama = header.index("Nama Usaha/ Perusahaan / Komplek Usaha")
idx_kec = header.index("kode_kecamatan")

data_per_kab = defaultdict(list)
for r in rows:
    nama = r[idx_nama]
    kec = r[idx_kec]
    kec = "" if kec is None else str(kec).strip()
    if nama and str(nama).strip():
        data_per_kab[kec].append(str(nama).strip())  # key "" = tanpa kecamatan spesifik

In [4]:
data_per_kab

defaultdict(list,
            {'1671050': ['MyeR',
              'Pertokoan Megaria',
              'BFI finance',
              'Bank Capital Indonesia',
              'Pee Wee Petshop'],
             '': ['Bank Utomo',
              'TBK Almond',
              'Dokter THT Hj Abla',
              'Toko Prestige Durian',
              'CLC',
              'Education Kids Center',
              'Kopi aba',
              'Auron Custom Jersey Apparel',
              'Dealer Mobil Chery',
              'Highfive Learning Center',
              'Bunda jaya',
              'PT. Berlian Propertindo',
              'PT. Waspada',
              'cv anugerah Multidana',
              'IGM',
              'CV. Defain Sinergi Energi Indonesia',
              'JM Group',
              '7 Days',
              'Kopi Canda Tawa',
              "Domino's Pizza",
              'Toko Miring',
              'Mie Gacoan',
              'Superindo',
              'Raja Gadai',
              'Ruang Guru',
  

In [6]:
out_dir = Path("../")
manifest = []
for kode_kab, names in data_per_kab.items():
    n_files = (len(names) - 1) // CHUNK + 1
    for i in range(0, len(names), CHUNK):
        chunk = names[i : i + CHUNK]
        fname = f"usaha_sulit_{kode_kab}_{i//CHUNK + 1}.sql"
        (out_dir / fname).write_text(build_sql(chunk, kode_kab), encoding="utf-8")
    manifest.append(
        f"{kode_kab}: {len(names)} rows -> {kode_kab}_1.sql ... {kode_kab}_{n_files}.sql"
    )
    print(manifest[-1])

print("Total files:", len(list(out_dir.glob("*.sql"))))

1671050: 5 rows -> 1671050_1.sql ... 1671050_1.sql
: 28 rows -> _1.sql ... _1.sql
1671060: 8 rows -> 1671060_1.sql ... 1671060_1.sql
1671061: 37 rows -> 1671061_1.sql ... 1671061_2.sql
1671051: 18 rows -> 1671051_1.sql ... 1671051_1.sql
1671031: 23 rows -> 1671031_1.sql ... 1671031_1.sql
1671011: 10 rows -> 1671011_1.sql ... 1671011_1.sql
1671021: 14 rows -> 1671021_1.sql ... 1671021_1.sql
1671040: 57 rows -> 1671040_1.sql ... 1671040_2.sql
1671041: 34 rows -> 1671041_1.sql ... 1671041_1.sql
1671080: 21 rows -> 1671080_1.sql ... 1671080_1.sql
1671081: 1 rows -> 1671081_1.sql ... 1671081_1.sql
Total files: 16
